In [ ]:
#| default_exp _trainer

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import torch
import matplotlib.pyplot as plt
from dreamer4._core import build_tiny_pinpad_dataset, make_tiny_dataloader
from dreamer4.dreamer4 import VideoTokenizer, DynamicsWorldModel
from dreamer4.trainers import VideoTokenizerTrainer, SimTrainer, cycle, BehaviorCloneTrainer

In [ ]:
CPU = False
neps = 20
device = 'cuda' if not CPU else 'cpu'
dataset, env = build_tiny_pinpad_dataset(device=device, episode_length=100, n=neps)

In [ ]:
import random
from typing import Optional, Literal, Dict, Any, List
from dreamer4.envs.pinpad import PinPad, MotionPlannerPinPad

import torch
from torch.utils.data import Dataset, TensorDataset

class NormalizeObsWrapper:
    """
    Wraps a PinPad env that returns (C,H,W) uint8/float in 0..255 and
    converts observations to float32 in [0,1]. Everything else is passthrough.
    """
    def __init__(self, env):
        self.env = env
        # expose gym-ish interface
        self.action_space = env.action_space
        self.observation_space = getattr(env, "observation_space", None)
        self.device = getattr(env, "device", "cpu")
        self.size = getattr(env, "size", (64, 64))
        self.task = getattr(env, "task", "three")

    def _norm(self, obs):
        # obs is a torch.Tensor (C,H,W) from your PinPad
        obs = obs.to(torch.float32)
        # If it looks like 0..255, scale to 0..1. Otherwise assume already normalized.
        if obs.max() > 1.0 or obs.min() < 0.0:
            obs = obs / 255.0
        return obs

    def reset(self, *args, **kwargs):
        obs = self.env.reset(*args, **kwargs)
        return self._norm(obs)

    def step(self, action):
        obs, r, done, truncated, info = self.env.step(action)
        return self._norm(obs), r, done, truncated, info

    # Optional: pass through render() etc.
    def render(self, *args, **kwargs):
        return self.env.render(*args, **kwargs)

class PinPadBCEpisodes(Dataset):
    def __init__(self, env, n_episodes=512, episode_length=16, use_motion_planner=True, return_length=16):
        self.samples = []
        H, W = env.size[0], env.size[1]

        self.episode_length=episode_length; self.return_length=return_length

        if use_motion_planner:
            mp = MotionPlannerPinPad(env)

        for _ in range(n_episodes):
            frames = []
            rewards = []
            actions = []

            obs = env.reset()                  # (C, H, W) float in 0..255
            for t in range(episode_length):
                frames.append(obs)              # store frame BEFORE action (standard)

                if use_motion_planner:
                    act = mp.sample()
                else:
                    act = env.action_space.sample()

                obs, r, done, _, _ = env.step(act)

                rewards.append(float(r))        # scalar
                actions.append(int(act))        # scalar int

                if done:                        # keep fixed length anyway
                    # pad the remainder by repeating last frame / zeros reward / no-op
                    for _pad in range(t + 1, episode_length):
                        frames.append(obs)
                        rewards.append(0.0)
                        actions.append(0)
                    break

            # Stack & reshape
            # frames: list of (C,H,W) -> (T,C,H,W) -> (C,T,H,W), normalize to [0,1]
            frames = torch.stack(frames, dim=0).float()  # (T,C,H,W)
            frames = frames.permute(1,0,2,3).contiguous()  # (C,T,H,W)
            frames = frames.detach().cpu()  # <-- keep CPU


            rewards = torch.tensor(rewards, dtype=torch.float32)  # (T,)
            discrete = torch.tensor(actions, dtype=torch.long).unsqueeze(-1)  # (T,1)

            # sanity
            assert frames.shape[1] == episode_length
            assert rewards.shape[0] == episode_length
            assert discrete.shape[:2] == (episode_length, 1)

            self.samples.append({
                "video": frames,                # (C,T,H,W) float[0,1]
                "rewards": rewards,             # (T,)
                "discrete_actions": discrete,   # (T,1)
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        # support both storage styles
        vid_key = "video_uint8" if "video_uint8" in s else "video"
        v = s[vid_key]                 # (C,T,H,W)
        r = s["rewards"]               # (T,)
        a = s["discrete_actions"]      # (T,1)

        C, T, H, W = v.shape
        K = self.return_length
        if K > T:
            raise ValueError(f"return_length ({K}) > episode_length ({T})")

        t0 = random.randint(0, T - K)  # inclusive range
        t1 = t0 + K

        # slice time dimension correctly
        v = v[:, t0:t1]                # (C,K,H,W)
        r = r[t0:t1]                   # (K,)
        a = a[t0:t1]                   # (K,1)

        # normalize if stored as uint8
        if v.dtype == torch.uint8:
            v = v.float().div_(255.0)

        # keep memory friendly
        v = v.contiguous()
        r = r.contiguous()
        a = a.contiguous()

        return {"video": v, "rewards": r, "discrete_actions": a}


class TokFromBCEpisodes(Dataset):
    """
    Sliding windows over bc_ds.samples[*]["video"] without extra storage.
    Assumes bc_ds.samples[i]["video"] is (C,T,H,W) on CPU.
    """
    def __init__(self, bc_ds, window=16, stride=1):
        self.bc = bc_ds
        self.window = window
        self.stride = stride
        self.T = bc_ds.episode_length
        self.starts_per_ep = max(0, (self.T - window)//stride + 1)

    def __len__(self):
        return len(self.bc.samples) * self.starts_per_ep

    def __getitem__(self, i):
        ep_idx = i // self.starts_per_ep
        start  = (i % self.starts_per_ep) * self.stride
        v = self.bc.samples[ep_idx]["video"]          # (C,T,H,W) CPU
        return v[:, start:start+self.window]          # (C,K,H,W) view

def build_pinpad_datasets_for_trainers(
    n_tok_episodes=512,
    n_bc_episodes=512,
    episode_length=16,
    device="cuda",
    use_motion_planner=True,
    normalize_in_env=True,   # <— new
):
    # base env
    base_env = PinPad('three', length=episode_length, extra_obs=False,
                      size=[64, 64], random_starting_pos=True, device=device)
    env = NormalizeObsWrapper(base_env) if normalize_in_env else base_env

    # ---- tokenizer dataset (VideoTokenizerTrainer still expects TensorDataset)
    vids = []
    mp_tok = MotionPlannerPinPad(env) if use_motion_planner else None
    for _ in range(n_tok_episodes):
        frames = []
        obs = env.reset()
        for t in range(episode_length):
            frames.append(obs)  # (C,H,W) already 0..1 if normalize_in_env
            act = mp_tok.sample() if mp_tok else env.action_space.sample()
            obs, r, done, _, _ = env.step(act)
            if done:
                for _pad in range(t + 1, episode_length):
                    frames.append(obs)
                break

        frames = torch.stack(frames, dim=1).contiguous()  # (C,T,H,W)
        if not normalize_in_env:
            frames = frames.float() / 255.0

        vids.append(frames)


    # ---- BC dataset (BehaviorCloneTrainer expects dict -> model(**batch))
    # Separate env so RNG/state differs a bit
    base_env_bc = PinPad('three', length=episode_length, extra_obs=False,
                         size=[64, 64], random_starting_pos=True, device=device)
    env_bc = NormalizeObsWrapper(base_env_bc) if normalize_in_env else base_env_bc
    bc_ds = PinPadBCEpisodes(env_bc, n_episodes=n_bc_episodes,
                             episode_length=episode_length,
                             use_motion_planner=use_motion_planner)


    tok_ds = TokFromBCEpisodes(bc_ds, window=16, stride=1)
    return tok_ds, bc_ds, env

# 1) Build datasets
tok_ds, bc_ds, env = build_pinpad_datasets_for_trainers(
    n_tok_episodes=512,
    n_bc_episodes=512,
    episode_length=100,
    device='cuda',
    use_motion_planner=True
)


In [ ]:
import torch
import matplotlib.pyplot as plt

def get_video_from_sample(sample):
    """Return a (C,T,H,W) or (T,C,H,W) tensor from dict/tuple/tensor samples."""
    if isinstance(sample, dict):
        v = sample.get("video", next((x for x in sample.values() if torch.is_tensor(x)), None))
    elif isinstance(sample, (list, tuple)):
        # common case: TensorDataset(videos) -> (video,)
        v = sample[0]
    else:
        v = sample
    if not torch.is_tensor(v):
        raise TypeError(f"Couldn't find a tensor video in sample of type {type(sample)}")
    return v

def canonicalize_cthw(v):
    """Ensure (C,T,H,W); convert dtype/range for visibility."""
    if v.ndim != 4:
        raise ValueError(f"Expected 4D video, got shape {tuple(v.shape)}")
    # If first dim isn't channels, assume (T,C,H,W) and permute
    if v.shape[0] not in (1,3):
        v = v.permute(1,0,2,3).contiguous()  # (T,C,H,W) -> (C,T,H,W)
    # Make float in [0,1] for viewing
    if v.dtype.is_floating_point:
        # if it looks zero-centered, de-normalize (heuristic)
        if v.min() < 0:
            v = (v * 0.5) + 0.5
        v = v.clamp(0,1)
    else:
        v = v.float() / 255.0
    return v

def peek_video(sample, title="sample", t=0):
    v = get_video_from_sample(sample)
    print(f"{title} raw:", v.shape, v.dtype, "min/max:", float(v.min()), float(v.max()))
    v = canonicalize_cthw(v)
    C,T,H,W = v.shape
    img = v[:, min(t,T-1)].permute(1,2,0).cpu().numpy()
    print(f"{title} canonical:", v.shape, v.dtype, "min/max:", float(v.min()), float(v.max()))
    plt.figure(figsize=(3,3)); plt.imshow(img); plt.title(f"{title} t={min(t,T-1)}"); plt.axis("off"); plt.show()
    return v

# Use on both datasets:
v_tok = peek_video(tok_ds[0], title="tok_ds[0]")
v_bc  = peek_video(bc_ds[0],  title="bc_ds[0]")


In [ ]:
# lpips_loss = 0.0
# dim, dim_latent = 64, 64
# tk = VideoTokenizer(
#     dim=dim,#16,
#     # encoder_depth = 1, # I think the paper has a different depths and does the time blocks every n (8?) layers
#     # decoder_depth = 1,
#     # time_block_every = 1,
#     dim_latent = dim_latent, #16,
#     patch_size = 16, #32
#     attn_dim_head = 16,
#     num_latent_tokens = 4,
#     lpips_loss_weight=lpips_loss,
# ).to(device)

# dynamics = DynamicsWorldModel(
#     video_tokenizer = tk,
#     dim = dim, #16,
#     dim_latent = dim_latent, #16,
#     max_steps = 64,
#     num_tasks = 1,
#     num_latent_tokens = 4, #1,
#     # depth = 1,
#     # time_block_every = 1,
#     # num_spatial_tokens = 1,
#     pred_orig_latent = True,
#     num_discrete_actions = env.action_space.n,
#     attn_dim_head = 16,
#     prob_no_shortcut_train = 0.1,
#     num_residual_streams = 1
# ).to(device)

# from chatgpt
dim, dim_latent, num_latent_tokens = 64, 64, 16
encoder_depth, decoder_depth = 2, 2

tk = VideoTokenizer(
    dim=dim,
    dim_latent=dim_latent,
    patch_size=4, #8,                 # was 16
    num_latent_tokens=num_latent_tokens, #8,          # was 4
    encoder_depth=encoder_depth,              # was default 4
    decoder_depth=decoder_depth,              # was default 4
    time_block_every=2,           # was 4
    attn_dim_head=32,             # was 16
    lpips_loss_weight=0.3,        # was 0.0
).to(device)

dynamics = DynamicsWorldModel(
    video_tokenizer=tk,
    dim=dim,
    dim_latent=dim_latent,
    max_steps=64,
    num_tasks=1,
    num_latent_tokens=num_latent_tokens,          # match tokenizer
    num_spatial_tokens=8,         # was 2 (too tiny)
    depth=4,                      # was default 4
    time_block_every=2,           # was 4
    pred_orig_latent=True,
    num_discrete_actions=env.action_space.n,
    attn_dim_head=32,
    prob_no_shortcut_train=0.4,   # give shortcut some presence
    num_residual_streams=1
).to(device)


In [ ]:


# 2) Train tokenizer (VideoTokenizerTrainer expects a TensorDataset and uses [0])
batch_size = 16
num_train_steps=100
tok_tr = VideoTokenizerTrainer(tk, tok_ds, batch_size=batch_size, num_train_steps=num_train_steps)

# 3) Train world model with BC (BehaviorCloneTrainer expects a dict batch and calls model(**batch))
bc_tr = BehaviorCloneTrainer(dynamics, bc_ds, batch_size=batch_size, num_train_steps=num_train_steps)

# 4) Evaluate / roll out as before (SimTrainer / eval_episodes, etc.)


In [ ]:
tok_tr.train()
bc_tr.train()

tok_tr()
bc_tr()

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

@torch.no_grad()
def robust_next_frame_eval(
    tokenizer,
    dynamics,
    video,             # (C,T,H,W)
    discrete_actions,  # (T, 1) or (T,)
    k=4,
    num_steps=4
):
    device = next(dynamics.parameters()).device
    
    # 1. Setup Video & Batch Dim
    if video.ndim == 4: video = video.unsqueeze(0) # (1, C, T, H, W)
    video = video.to(device)
    B, C, T, H, W = video.shape

    # 2. Setup Actions (The Fix)
    if discrete_actions is not None:
        da = discrete_actions.to(device)
        
        # --- FIXED DIMENSION LOGIC ---
        if da.ndim == 1: 
            # Case: (T,) -> (1, T, 1)
            da = da.unsqueeze(0).unsqueeze(-1) 
        elif da.ndim == 2:
            # Case: (T, 1) -> (1, T, 1) <-- This was missing!
            da = da.unsqueeze(0)
        # -----------------------------
        
        # Slice for time: we need actions 0...k to predict states 1...k+1
        da_input = da[:, :k+1] 
    else:
        da_input = None

    # 3. Tokenize History (0 to k-1)
    lat_past = tokenizer.tokenize(video[:, :, :k]) # (1, k, n, d)
    if lat_past.ndim == 4: lat_past = lat_past.unsqueeze(2) # (1, k, 1, n, d)
    
    # 4. Init Target Latent (Frame k) as Noise
    n, d = lat_past.shape[-2:]
    lat_next = torch.randn((1, 1, 1, n, d), device=device)

    # 5. Schedule Loop
    step_size = dynamics.max_steps // num_steps
    levels = torch.arange(0, dynamics.max_steps, step_size, device=device, dtype=torch.long)
    
    for level in levels:
        lat_in = torch.cat([lat_past, lat_next], dim=1)

        sig_next = level.view(1, 1)
        sig_past = torch.full((1, k), dynamics.max_steps - 1, device=device, dtype=torch.long)
        sig_full = torch.cat([sig_past, sig_next], dim=1)

        pred, _ = dynamics.forward(
            latents=lat_in,
            signal_levels=sig_full,
            step_sizes=step_size,
            discrete_actions=da_input,
            latent_is_noised=True,
            return_pred_only=True,
            return_intermediates=True,
            latent_has_view_dim=True,
        )
        
        if isinstance(pred, tuple): pred = pred[0]
        model_out = pred[:, -1:]
        
        if dynamics.pred_orig_latent:
            t_curr = level.float() / dynamics.max_steps
            t_curr = torch.clamp(t_curr, 0.0, 0.999)
            flow = (model_out - lat_next) / (1.0 - t_curr)
        else:
            flow = model_out
            
        dt = step_size / dynamics.max_steps
        lat_next = lat_next + flow * dt

# 6. Decode
    pred_lat = lat_next[0, 0, 0] 
    pred_video = tokenizer.decode(pred_lat.unsqueeze(0).unsqueeze(0), height=H, width=W)
    
    return {
        # FIX: Use [0, :, 0] to keep all channels (3, H, W)
        "pred_frame": pred_video[0, :, 0].cpu(), 
        "true_frame": video[0, :, k].cpu()
    }

def viz_robust_frame(tokenizer, dynamics, video, discrete_actions, k=4, num_steps=4, title=None):
    # Call the new robust evaluator
    out = robust_next_frame_eval(tokenizer, dynamics, video, discrete_actions, k, num_steps)
    
    true_f = out['true_frame']
    pred_f = out['pred_frame']
    
    # Calculate MSE for the title
    mse = F.mse_loss(true_f, pred_f).item()
    
    # Convert to numpy for plotting (C, H, W) -> (H, W, C)
    def to_img(x): return x.permute(1,2,0).clamp(0,1).numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(to_img(true_f))
    axes[0].set_title(f"True Frame (t={k})")
    axes[0].axis("off")
    
    axes[1].imshow(to_img(pred_f))
    axes[1].set_title(f"Pred | MSE: {mse:.5f}")
    axes[1].axis("off")
    
    if title: fig.suptitle(title)
    plt.show()
    
    return mse


sample = bc_ds[0]
video = sample["video"].to(tk.device)             # (C,T,H,W) or (1,C,T,H,W)
acts  = sample.get("discrete_actions").to(tk.device)
rews  = sample.get("rewards").to(tk.device)

mse = viz_robust_frame(
    tokenizer=tk,
    dynamics=dynamics,
    video=video,
    discrete_actions=acts,
    k=4,
    num_steps=4,
    title="Next-frame via denoising schedule"
)
print("Next-frame MSE:", mse)

In [ ]:
# show all 16 frames of the video in a grid
def plot_video_grid(video, rows=4, cols=4, title="Video Grid"):
    action_map = {0: 'none', 2: 'up', 1: 'down', 3: 'right', 4:'left'}

    v = canonicalize_cthw(get_video_from_sample(video))  # (C,T,H,W)
    C,T,H,W = v.shape
    assert rows * cols >= T, "Not enough grid cells for all frames"
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
    for i in range(rows):
        for j in range(cols):
            idx = i * cols + j
            axes[i,j].axis("off")
            if idx < T:
                img = v[:, idx].permute(1,2,0).cpu().numpy()
                axes[i,j].imshow(img)
                title=f"t={idx} Action: {action_map.get(sample['discrete_actions'][idx].item(), 'N/A')}"
                axes[i,j].set_title(title)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_video_grid(sample, rows=4, cols=4, title="Sample Video from BC Dataset")

In [ ]:
import torch, numpy as np
import torch.nn.functional as F

@torch.no_grad()
def eval_open_loop_k_steps(tokenizer, dynamics, video, k=4, H=8, actions=None, rewards=None):
    """
    Predict H future frames given a rolling window of the last k frames.
    Keeps the context fixed at length k (so model input is always k+1).
    Returns per-step MSE and mean MSE.
    """
    device = next(dynamics.parameters()).device

    # standardize to (1,C,T,H,W)
    if video.ndim == 4:
        video = video.unsqueeze(0)
    elif video.ndim != 5:
        raise ValueError(f"video must be (C,T,H,W) or (1,C,T,H,W); got {tuple(video.shape)}")
    video = video.to(device)
    B,C,T,VH,VW = video.shape
    if B != 1: raise ValueError("Only B=1 supported for this eval.")
    if T <= k: raise ValueError(f"Need T > k (T={T}, k={k}).")
    if H > (T - k): H = T - k  # clip horizon if needed

    # optional A/R to (1,T,...) with right padding if short
    def _prep_opt(x, need_len):
        if x is None: return None
        x = x.to(device)
        if x.ndim == 1: x = x.unsqueeze(0)            # (1,T)
        elif x.ndim == 2 and x.shape[0] != 1: x = x.unsqueeze(0)  # (1,T,NA?)
        if x.shape[1] < need_len:
            pad = torch.zeros((x.shape[0], need_len - x.shape[1], *x.shape[2:]),
                              dtype=x.dtype, device=x.device)
            x = torch.cat([x, pad], dim=1)
        return x

    actions = _prep_opt(actions, need_len=T)
    rewards = _prep_opt(rewards, need_len=T)

    # tokenize initial history (first k)
    lat_past = tokenizer.tokenize(video[:, :, :k]).unsqueeze(2)  # (1,k,1,n,d)
    n, d = lat_past.shape[-2], lat_past.shape[-1]

    per_step_mse = []

    for step in range(H):
        # always condition on last k -> input length is k+1
        lat_slot = torch.randn((1,1,1,n,d), device=device)
        lat_in = torch.cat([lat_past, lat_slot], dim=1)          # (1,k+1,1,n,d)

        # align A/R to last k (then pad one)
        da = None
        if actions is not None:
            a_k = actions[:, step:step+k]                         # (1,k,NA?) or (1,k)
            pad = torch.zeros_like(a_k[:, :1])                    # (1,1,...)
            da = torch.cat([a_k, pad], dim=1)                     # (1,k+1,...)

        rw = None
        if rewards is not None:
            r_k = rewards[:, step:step+k]                         # (1,k)
            rw  = torch.cat([r_k, torch.zeros_like(r_k[:, :1])], dim=1)  # (1,k+1)

        # inference schedule for exactly k+1 tokens
        step_size = torch.tensor(1, device=device)
        signal_levels = torch.full((1, k+1), dynamics.max_steps - 1,
                                   dtype=torch.long, device=device)

        # predict next latent (one denoise step)
        pred, _ = dynamics.forward(
            latents=lat_in,
            signal_levels=signal_levels,
            step_sizes=step_size,
            rewards=rw,
            discrete_actions=da,
            latent_is_noised=True,
            return_pred_only=True,
            return_intermediates=True,
            latent_has_view_dim=True,
        )
        if isinstance(pred, tuple): pred = pred[0]                # strip proprio if present
        pred_next_lat = pred[:, -1, 0]                            # (1,n,d)

        # decode and compare to ground truth at t = k+step
        pred_frame = tokenizer.decode(pred_next_lat, height=VH, width=VW)[0, :, 0]
        true_frame = video[0, :, k + step]
        per_step_mse.append(F.mse_loss(pred_frame, true_frame).item())

        # roll history window: drop oldest, append GT next latent
        gt_next_lat = tokenizer.tokenize(video[:, :, k + step : k + step + 1])  # (1,1,n,d)
        lat_past = torch.cat([lat_past[:, 1:], gt_next_lat.unsqueeze(2)], dim=1)  # keep length k

    return {
        "per_step_mse": np.array(per_step_mse, dtype=np.float32),
        "mean_mse": float(np.mean(per_step_mse)),
        "k": k,
        "H": H,
        "T": int(T),
    }


import torch, numpy as np
import matplotlib.pyplot as plt

sample = bc_ds[0]
video = sample["video"]            # (C,T,H,W)
acts  = sample.get("discrete_actions")
rews  = sample.get("rewards")

print("video shape:", tuple(video.shape))  # helpful debug
k, H = 4, 8

out = eval_open_loop_k_steps(
    tokenizer=tk,
    dynamics=dynamics,
    video=video,
    k=k,
    H=H,                 # will auto-clip to T-k if needed
    actions=acts,
    rewards=rews
)

print("T, k, (possibly clipped) H:", out["T"], out["k"], out["H"])
print("Per-step MSE:", out["per_step_mse"], "Mean:", out["mean_mse"])


# quick plot
plt.figure(figsize=(5,3))
plt.plot(range(1, H+1), out["per_step_mse"], marker="o")
plt.xlabel("Prediction step (t = k + step)")
plt.ylabel("MSE")
plt.title(f"Open-loop {H}-step MSE (k={k})")
plt.tight_layout(); plt.show()

In [ ]:
# --- EVAL HARNESS FOR PINPAD + DYNAMICS ---

import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

@torch.no_grad()
def run_env_eval(
    env,
    dynamics,                # DynamicsWorldModel (already paired with tokenizer)
    tokenizer=None,          # VideoTokenizer or None
    episodes=5,
    max_steps=64,            # steps per episode (caps rollout inside interact_with_env)
    env_is_vectorized=False,
    show_first_k=8,          # how many frames to visualize
    show=True                # set False for headless metric-only
):
    device = next(dynamics.parameters()).device

    # put model in eval for stable behavior
    was_training = dynamics.training
    dynamics.eval()

    ep_stats = []
    visuals = []  # optional (video, recon) pairs for first few episodes

    for ep in range(episodes):
        exp = dynamics.interact_with_env(
            env,
            max_timesteps=max_steps,
            env_is_vectorized=env_is_vectorized,
            use_time_cache=True,
            store_agent_embed=False,
            store_old_action_unembeds=False
        )

        # rewards: (B, T) where B=1 here
        rewards = exp.rewards.squeeze(0) if exp.rewards is not None else torch.zeros(0, device=device)
        total_r = rewards.sum().item()
        ep_len  = rewards.numel()
        # PinPad marks success with +10 reward on success step
        success = (rewards.max().item() >= 9.99) if rewards.numel() > 0 else False

        stat = dict(reward=total_r, length=ep_len, success=bool(success))
        ep_stats.append(stat)

        if show and exp.video is not None and tokenizer is not None:
            # exp.video: (B,C,T,H,W); exp.latents: (B,T, [V=1], N, D) or (B,T,N,D)
            video = exp.video[0].detach().cpu()                      # (C,T,H,W)
            latents = exp.latents.to(device)
            # decode latents back to video for comparison
            C, T, H, W = video.shape
            lat_for_dec = latents
            if lat_for_dec.ndim == 5:  # (B,T,1,N,D) -> (B,T,N,D)
                lat_for_dec = lat_for_dec[..., 0, :, :]
            recon = tokenizer.decode(lat_for_dec, height=H, width=W)[0].detach().cpu()  # (C,T,H,W)
            visuals.append((video, recon))

            # visualize a few frames inline
            t_show = min(T, show_first_k)
            fig, axes = plt.subplots(2, t_show, figsize=(3*t_show, 6))
            for i in range(t_show):
                axes[0, i].imshow(video[:, i].permute(1,2,0).clamp(0,1).numpy())
                axes[0, i].set_title(f"ep {ep} | orig t={i}")
                axes[0, i].axis("off")

                axes[1, i].imshow(recon[:, i].permute(1,2,0).clamp(0,1).numpy())
                axes[1, i].set_title(f"ep {ep} | recon t={i}")
                axes[1, i].axis("off")

            fig.suptitle(f"Reward={total_r:.2f} | Len={ep_len} | Success={success}")
            plt.tight_layout()
            display(fig)
            plt.close(fig)

    # restore training state
    dynamics.train(was_training)

    # aggregate metrics
    rewards = [s["reward"] for s in ep_stats]
    lengths = [s["length"] for s in ep_stats]
    successes = [s["success"] for s in ep_stats]

    summary = dict(
        episodes=episodes,
        avg_reward=float(np.mean(rewards)) if rewards else 0.0,
        avg_length=float(np.mean(lengths)) if lengths else 0.0,
        success_rate=float(np.mean(successes)) if successes else 0.0,
        per_episode=ep_stats,
        visuals=visuals  # list of (video, recon), only when show=True & tokenizer provided
    )

    print(
        f"[EVAL] episodes={episodes} | "
        f"avg_reward={summary['avg_reward']:.3f} | "
        f"avg_len={summary['avg_length']:.1f} | "
        f"success_rate={summary['success_rate']*100:.1f}%"
    )
    return summary


In [ ]:
# Main training loop
import torch, numpy as np
import matplotlib.pyplot as plt


for _ in range(10):
    # ... (your existing data loading code) ...
    video = sample["video"].to(tk.device)
    acts  = sample.get("discrete_actions").to(tk.device)

    # REPLACE THIS CALL:
    # out = viz_next_frame_with_schedule(...) 
    
    # WITH THIS:
    mse = viz_robust_frame(
        tokenizer=tk,
        dynamics=dynamics,
        video=video,
        discrete_actions=acts,
        k=4,
        num_steps=4,
        title="Robust Next-Frame Eval"
    )
    print("Next-frame MSE:", mse)

    # set to training mode
    tk.train()
    dynamics.train()

    tok_tr()
    bc_tr()

    # grab one episode from your BC dataset
    sample = bc_ds[0]  # depends on your PinPadBCEpisodes; adapt keys accordingly

    # if your BC dataset returns a dict:
    video = sample["video"].to(tk.device)                # (C,T,H,W) in [0,1]
    acts  = sample.get("discrete_actions").to(tk.device) # (T,1) optional
    rews  = sample.get("rewards").to(tk.device)          # (T,)   optional

    # WITH THIS:
    mse = viz_robust_frame(
        tokenizer=tk,
        dynamics=dynamics,
        video=video,
        discrete_actions=acts,
        k=4,
        num_steps=4,
        title="Robust Next-Frame Eval"
    )

    print("Next-frame MSE:", mse)


    k, H = 4, 8
    mets = eval_open_loop_k_steps(
        tokenizer=tk,
        dynamics=dynamics,
        video=video,
        k=k,
        H=H,                 # will auto-clip to T-k if needed
        actions=acts,
        rewards=rews
    )

    print("T, k, (possibly clipped) H:", mets["T"], mets["k"], mets["H"])
    print("Per-step MSE:", mets["per_step_mse"], "Mean:", mets["mean_mse"])



In [ ]:
def check_tokenizer_fidelity(tokenizer, video):
    """
    Sanity check: Can we reconstruct the pin from just 8 tokens?
    Handles shape correction automatically.
    """
    device = next(tokenizer.parameters()).device
    video = video.to(device)
    
    # FIX: Ensure (1, C, T, H, W)
    if video.ndim == 4:
        video = video.unsqueeze(0)
        
    tokenizer.eval()
    with torch.no_grad():
        # Encode
        latents = tokenizer.tokenize(video) # (1, T, N, D)
        # Decode
        recon = tokenizer.decode(latents, height=64, width=64)
    
    # Visual check
    # Pick a frame in the middle to see movement
    t_idx = min(video.shape[2]-1, 5)
    
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    
    # Clamp and Permute for matplotlib (C, H, W) -> (H, W, C)
    orig_img = video[0, :, t_idx].permute(1,2,0).cpu().clamp(0,1).numpy()
    recon_img = recon[0, :, t_idx].permute(1,2,0).cpu().clamp(0,1).numpy()
    
    axes[0].imshow(orig_img)
    axes[0].set_title(f"Original (t={t_idx})")
    axes[0].axis('off')
    
    axes[1].imshow(recon_img)
    axes[1].set_title("Reconstructed")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# Run the fixed sanity check
check_tokenizer_fidelity(tk, video)

In [ ]:
if DO_PLOT:=True:
    # quick plot
    plt.figure(figsize=(5,3))
    plt.plot(range(1, H+1), mets["per_step_mse"], marker="o")
    plt.xlabel("Prediction step (t = k + step)")
    plt.ylabel("MSE")
    plt.title(f"Open-loop {H}-step MSE (k={k})")
    plt.tight_layout(); plt.show(block=False)

    # assuming you already have:
    # env          = PinPad(...) or your wrapped env (normalizes to [0,1])
    # tokenizer    = VideoTokenizer(...); tokenizer.eval()
    # dynamics     = DynamicsWorldModel(..., video_tokenizer=tokenizer, ...); dynamics.eval()
    # (optionally load weights for both)


In [ ]:
if DO_EVAL:=True:
    # set to eval
    tk.eval()
    dynamics.eval()

    summary = run_env_eval(
        env=env,
        dynamics=dynamics,
        tokenizer=tk,   # pass None to skip recon visuals
        episodes=5,
        max_steps=100,
        show_first_k=8,
        show=False
    )

    mets.update(summary)

    for k,v in mets.items():
        print(f"{k} {v}")


    # visualize out["pred_frame"] vs out["target_frame"]


In [ ]:


summary = run_env_eval(
    env=env,
    dynamics=dynamics,
    tokenizer=tk,   # pass None to skip recon visuals
    episodes=1,
    max_steps=100,
    show_first_k=100,
    show=True
)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()